# TrustVLA RunPod LIBERO/OpenVLA Pilot

Use this notebook on a Linux GPU/SIM machine such as RunPod. It is not meant for local Mac execution unless LIBERO and MuJoCo/robosuite are already installed.

In [ ]:
from pathlib import Path
import subprocess
import os

PROJECT_DIR = Path.cwd()
SUITE = "libero_object"
LIMIT = 10
HF_LOCAL_DIR = "/workspace/LIBERO-datasets"
MODEL_PATH = "openvla/openvla-7b"
DEVICE = "cuda:0"
RUN_ROLLOUTS = False  # Set True only after seed annotation is complete.

def run(cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check, cwd=PROJECT_DIR)

print("Project:", PROJECT_DIR)

## 1. Environment Check

If `libero` is missing, install LIBERO before continuing to real rollout cells.

In [ ]:
run(["python", "-m", "trustvla.cli", "doctor"], check=False)

## 2. Selective Hugging Face Download

This downloads only one LIBERO suite, not the full dataset mirror.

In [ ]:
run([
    "python", "-m", "trustvla.cli", "download-libero-hf",
    "--suite", SUITE,
    "--local-dir", HF_LOCAL_DIR,
])

## 3. Export LIBERO Seed Draft

This requires LIBERO to be installed. The output needs manual annotation before benchmark generation.

In [ ]:
seed_path = f"data/{SUITE}_seed_draft.json"
run([
    "python", "-m", "trustvla.cli", "export-libero-seeds",
    "--suite", SUITE,
    "--limit", str(LIMIT),
    "--out", seed_path,
])
print("Annotate:", seed_path)

## 4. Generate TrustVLA-Pairs After Annotation

Run this only after filling target objects, distractors, absent objects, ambiguous targets, and hazards in the seed draft.

In [ ]:
benchmark_path = f"runs/{SUITE}/trustvla_pairs.jsonl"
run([
    "python", "-m", "trustvla.cli", "generate",
    "--seed-tasks", seed_path,
    "--out", benchmark_path,
])
print("Benchmark:", benchmark_path)

## 5. OpenVLA Rollouts

Set `RUN_ROLLOUTS = True` in the config cell only after the benchmark file looks correct.

In [ ]:
if RUN_ROLLOUTS:
    run([
        "python", "-m", "trustvla.cli", "run-openvla-libero",
        "--benchmark", benchmark_path,
        "--out", f"runs/{SUITE}/openvla_rollouts.jsonl",
        "--model-path", MODEL_PATH,
        "--suite", SUITE,
        "--device", DEVICE,
        "--trace-dir", f"runs/{SUITE}/traces/openvla",
    ])
else:
    print("Skipping rollout. Set RUN_ROLLOUTS=True after annotation is complete.")

## 6. Detect Events And Compare

Run after real rollout JSONL files exist.

In [ ]:
if RUN_ROLLOUTS:
    run([
        "python", "-m", "trustvla.cli", "detect-rollout-events",
        "--benchmark", benchmark_path,
        "--rollouts", f"runs/{SUITE}/openvla_rollouts.jsonl",
        "--out", f"runs/{SUITE}/openvla_rollouts.detected.jsonl",
    ])
else:
    print("Skipping detection until real rollouts exist.")